### we will fine tune our best two models

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer,OrdinalEncoder
from sklearn.model_selection import train_test_split

In [2]:
%pip install mlflow dagshub

Note: you may need to restart the kernel to use updated packages.


In [3]:
import mlflow

In [6]:
import dagshub
import mlflow

# Authenticate with your DagsHub repository
dagshub.init(
    repo_owner="aryann13",
    repo_name="Swiggy-Delivery-Time-Prediction",
    mlflow=True,
)

# Set the tracking server
mlflow.set_tracking_uri(
    "https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow"
)


Accessing as aryann13

Initialized MLflow to track repo "aryann13/Swiggy-Delivery-Time-Prediction"

Repository aryann13/Swiggy-Delivery-Time-Prediction initialized!

In [7]:
# mlflow experiment name
mlflow.set_experiment("Exp 4 - Final Model HP tuning")

<Experiment: artifact_location='mlflow-artifacts:/fac2861a61244de7ba66eb3ec75528fc', creation_time=1789192525063, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1789192525063, lifecycle_stage='active', name='Exp 4 - Final Model HP tuning', tags={}, trace_location=None, workspace='default'>

In [8]:
from sklearn import set_config

set_config(transform_output = "pandas")

### Load the data

In [9]:
df = pd.read_csv('swiggy.csv')
df

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45588,0x7c09,JAPRES04DEL01,30,4.8,26.902328,75.794257,26.912328,75.804257,24-03-2022,11:35:00,11:45:00,conditions Windy,High,1,Meal,motorcycle,0,No,Metropolitian,(min) 32
45589,0xd641,AGRRES16DEL01,21,4.6,0.000000,0.000000,0.070000,0.070000,16-02-2022,19:55:00,20:10:00,conditions Windy,Jam,0,Buffet,motorcycle,1,No,Metropolitian,(min) 36
45590,0x4f8d,CHENRES08DEL03,30,4.9,13.022394,80.242439,13.052394,80.272439,11-03-2022,23:50:00,00:05:00,conditions Cloudy,Low,1,Drinks,scooter,0,No,Metropolitian,(min) 16
45591,0x5eee,COIMBRES11DEL01,20,4.7,11.001753,76.986241,11.041753,77.026241,07-03-2022,13:35:00,13:40:00,conditions Cloudy,High,0,Snack,motorcycle,1,No,Metropolitian,(min) 26


In [12]:
df = pd.read_csv("cleaned_data.csv")
df


,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance,distance_type
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,sunny,high,2,snack,motorcycle,0.0,no,urban,24,INDO,19,3,saturday,1,15.0,11.0,morning,3.025149,short
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,BANG,25,3,friday,0,5.0,19.0,evening,20.183530,very_long
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,BANG,19,3,saturday,1,15.0,8.0,morning,1.552758,short
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,COIMB,5,4,tuesday,0,10.0,18.0,evening,7.790401,medium
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,CHEN,26,3,saturday,1,15.0,13.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,JAPRES04DEL01,30.0,4.8,26.902328,75.794257,26.912328,75.804257,24-03-2022,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,JAP,24,3,thursday,0,10.0,11.0,morning,1.489846,short
45498,AGRRES16DEL01,21.0,4.6,NaN,NaN,NaN,NaN,16-02-2022,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,36,AGR,16,2,wednesday,0,15.0,19.0,evening,NaN,NaN
45499,CHENRES08DEL03,30.0,4.9,13.022394,80.242439,13.052394,80.272439,11-03-2022,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,CHEN,11,3,friday,0,15.0,23.0,night,4.657195,short
45500,COIMBRES11DEL01,20.0,4.7,11.001753,76.986241,11.041753,77.026241,07-03-2022,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,COIMB,7,3,monday,0,5.0,13.0,afternoon,6.232393,medium


In [13]:
# drop columns not required for model input

columns_to_drop = [
    "rider_id",
    "restaurant_latitude",
    "restaurant_longitude",
    "delivery_latitude",
    "delivery_longitude",
    "order_date",
    "order_time_hour",
    "order_day",
    "city_name",
    "order_day_of_week",
    "order_month",
]

df.drop(columns=columns_to_drop, inplace=True)
df


,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
45498,21.0,4.6,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,36,0,15.0,evening,NaN,NaN
45499,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
45500,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [16]:
missing_cols = (
    df.isna().any(axis=0)
    .loc[lambda x: x]
    .index
)
missing_cols

Index(['age', 'ratings', 'weather', 'traffic', 'multiple_deliveries',
       'festival', 'city_type', 'pickup_time_minutes', 'order_time_of_day',
       'distance', 'distance_type'],
      dtype='str')

In [14]:
temp_df = df.copy().dropna()

In [17]:
#split into X and y
# split into X and y
X = temp_df.drop(columns="time_taken")
y = temp_df["time_taken"]
print("Remaining rows after dropping NA:", temp_df.shape[0])

Remaining rows after dropping NA: 37695


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("The size of train data is", X_train.shape)
print("The shape of test data is", X_test.shape)

The size of train data is (30156, 15)
The shape of test data is (7539, 15)


In [19]:
# missing values in train data

X_train.isna().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [20]:
# transform target column

pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [21]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [24]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [25]:
nominal_cat_cols

['weather',
 'type_of_order',
 'type_of_vehicle',
 'festival',
 'city_type',
 'is_weekend',
 'order_time_of_day']

In [27]:
# unique categories the ordinal columns

for col in ordinal_cat_cols:
    print(col,X_train[col].unique())

traffic <ArrowStringArray>
['jam', 'medium', 'high', 'low']
Length: 4, dtype: str
distance_type <ArrowStringArray>
['medium', 'short', 'long', 'very_long']
Length: 4, dtype: str


In [26]:
X_train.isna().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [28]:
# build a preprocessor

preprocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",
                                     sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                      encoded_missing_value=-999,
                                      handle_unknown="use_encoded_value",
                                      unknown_value=-1), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,verbose_feature_names_out=False)


preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('nominal_encode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for featur

In [29]:
# build processing pipeline (no imputers needed because NA was dropped)
processing_pipeline = Pipeline(steps=[("preprocess", preprocessor)])

# fit and transform
X_train_trans = processing_pipeline.fit_transform(X_train)
X_test_trans = processing_pipeline.transform(X_test)

print("Transformed train shape:", X_train_trans.shape)
print("Transformed test shape:", X_test_trans.shape)


Transformed train shape: (30156, 25)
Transformed test shape: (7539, 25)


In [31]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.compose import TransformedTargetRegressor


In [32]:
def objective_rf(trial):
  with mlflow.start_run(nested=True):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 300),
        "max_depth": trial.suggest_int("max_depth", 5, 25),
        "max_features": trial.suggest_categorical(
            "max_features", [None, "sqrt", "log2"]
        ),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 6),
        "max_samples": trial.suggest_float("max_samples", 0.6, 1.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    # log trial parameters
    mlflow.log_params(params)

    # build model with transformed target
    rf = RandomForestRegressor(**params)
    model = TransformedTargetRegressor(regressor=rf, transformer=pt)

    # 5-fold cross-validation on negative MAE
    cv_score = cross_val_score(
        model,
        X_train_trans,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    )

    mean_score = -cv_score.mean()
    mlflow.log_metric("cross_val_error", mean_score)

    return mean_score


In [33]:
# create and run study
study_rf = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="Tuned_Random_Forest"):
  study_rf.optimize(objective_rf, n_trials=20, n_jobs=-1, show_progress_bar=True)

  # log best parameters and best CV score
  mlflow.log_params(study_rf.best_params)
  mlflow.log_metric("best_cv_score", study_rf.best_value)

print("Best RF Parameters:", study_rf.best_params)
print("Best RF CV MAE:", study_rf.best_value, "minutes")


[I 2026-09-12 11:57:05,619] A new study created in memory with name: no-name-eed363e5-70b5-4976-a182-fe1bc9a1b22e


  0%|          | 0/20 [00:00<?, ?it/s]

🏃 View run trusting-shark-661 at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/673c899f4299440dad7796e55b8a4dcc
🧪 View experiment at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4
[I 2026-09-12 11:57:33,121] Trial 14 finished with value: 3.9869599219804797 and parameters: {'n_estimators': 99, 'max_depth': 7, 'max_features': 'sqrt', 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_samples': 0.6763895498433479}. Best is trial 14 with value: 3.9869599219804797.
🏃 View run gifted-dog-146 at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/9feff6d6417b4b3ea01e43cf4cfbafab
🧪 View experiment at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4
🏃 View run adaptable-horse-20 at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/6a166bfbeac8426ebaf3f1763749d0f9
🧪 View experiment at: https://dagshub.

In [34]:
def objective_xgb(trial):
  with mlflow.start_run(nested=True):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 250),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),  # identical to min_split_gain
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 50.0),
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
    }

    # log trial parameters
    mlflow.log_params(params)

    # build model with transformed target
    xgb = XGBRegressor(**params)
    model = TransformedTargetRegressor(regressor=xgb, transformer=pt)

    # 5-fold cross-validation on negative MAE
    cv_score = cross_val_score(
        model,
        X_train_trans,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    )

    mean_score = -cv_score.mean()
    mlflow.log_metric("cross_val_error", mean_score)

    return mean_score


In [35]:
# create and run study
study_xgb = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="Tuned_XGBoost"):
  study_xgb.optimize(
      objective_xgb, n_trials=20, n_jobs=-1, show_progress_bar=True
  )

  # log best parameters and best CV score
  mlflow.log_params(study_xgb.best_params)
  mlflow.log_metric("best_cv_score", study_xgb.best_value)

print("Best XGB Parameters:", study_xgb.best_params)
print("Best XGB CV MAE:", study_xgb.best_value, "minutes")


[I 2026-09-12 11:59:52,556] A new study created in memory with name: no-name-e69ae127-e8ad-45a6-9ea7-37ca5c2924e0


  0%|          | 0/20 [00:00<?, ?it/s]

🏃 View run placid-rook-140 at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/3d3668d20be3470694304396aa278521
🧪 View experiment at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4
🏃 View run delightful-auk-695 at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/5241eae59409486a87324a489556a3bb
🧪 View experiment at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4
🏃 View run rogue-fox-587 at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/6cfeb80631604967adc2d60929792feb
🧪 View experiment at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4
🏃 View run honorable-chimp-237 at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/0f448b148a40474caa899f0bebca25d5
🧪 View experiment at: https://dagshub.com/aryann13/Swiggy-Deliver

In [36]:
# 1. Optimization History for Random Forest
optuna.visualization.plot_optimization_history(study_rf)


In [37]:
# 2. Hyperparameter Importance for Random Forest
optuna.visualization.plot_param_importances(study_rf)


In [38]:
# 3. Slice Plot for Random Forest
optuna.visualization.plot_slice(study_rf)


In [39]:
# 1. Optimization History for XGBoost
optuna.visualization.plot_optimization_history(study_xgb)


In [40]:
# 2. Hyperparameter Importance for XGBoost
optuna.visualization.plot_param_importances(study_xgb)


#### Step 1: Stacking Regressor Tuning

In [41]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor


In [42]:
# Instantiate best RF and best XGB using the optimal parameters found earlier
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=42)
best_xgb = XGBRegressor(
    **study_xgb.best_params, tree_method="hist", random_state=42
)


In [43]:
def objective_stacking(trial):
  with mlflow.start_run(nested=True):
    meta_model_name = trial.suggest_categorical("model", ["LR", "KNN", "DT"])

    if meta_model_name == "LR":
      meta = LinearRegression()

    elif meta_model_name == "KNN":
      n_neighbors_knn = trial.suggest_int("n_neighbors_knn", 1, 15)
      weights_knn = trial.suggest_categorical(
          "weights_knn", ["uniform", "distance"]
      )
      meta = KNeighborsRegressor(
          n_neighbors=n_neighbors_knn, weights=weights_knn, n_jobs=-1
      )

    elif meta_model_name == "DT":
      max_depth_dt = trial.suggest_int("max_depth_dt", 1, 10)
      min_samples_split_dt = trial.suggest_int("min_samples_split_dt", 2, 10)
      min_samples_leaf_dt = trial.suggest_int("min_samples_leaf_dt", 1, 10)
      meta = DecisionTreeRegressor(
          max_depth=max_depth_dt,
          min_samples_split=min_samples_split_dt,
          min_samples_leaf=min_samples_leaf_dt,
          random_state=42,
      )

    mlflow.log_param("meta_model_name", meta_model_name)

    # assemble the stacking regressor
    stacking_reg = StackingRegressor(
        estimators=[("rf", best_rf), ("xgb", best_xgb)],
        final_estimator=meta,
        cv=5,
        n_jobs=-1,
    )

    # wrap with transformed target
    model = TransformedTargetRegressor(regressor=stacking_reg, transformer=pt)

    # fit on train and evaluate on test
    model.fit(X_train_trans, y_train)
    y_pred_test = model.predict(X_test_trans)

    error = mean_absolute_error(y_test, y_pred_test)
    mlflow.log_metric("MAE", error)

    return error


In [44]:
study_stacking = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="Stacking_Regressor_Tuning"):
  study_stacking.optimize(
      objective_stacking, n_trials=20, n_jobs=-1, show_progress_bar=True
  )

  mlflow.log_params(study_stacking.best_params)
  mlflow.log_metric("best_score", study_stacking.best_value)

print("Best Meta-Learner:", study_stacking.best_params)
print("Best Stacking MAE:", study_stacking.best_value, "minutes")


[I 2026-09-12 12:28:40,457] A new study created in memory with name: no-name-09c94382-891d-4460-a369-944698abcb41


  0%|          | 0/20 [00:00<?, ?it/s]

🏃 View run languid-grub-524 at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/9d0b575f7a3c42c495af9c3bcccc1205
🧪 View experiment at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4
[I 2026-09-12 12:31:28,245] Trial 13 finished with value: 3.0520134758073443 and parameters: {'model': 'DT', 'max_depth_dt': 10, 'min_samples_split_dt': 9, 'min_samples_leaf_dt': 4}. Best is trial 13 with value: 3.0520134758073443.
🏃 View run angry-hen-754 at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/5af88c1961c047bf9dcfbb96ec7ea36c
🧪 View experiment at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4
[I 2026-09-12 12:31:28,775] Trial 6 finished with value: 3.1142937294860804 and parameters: {'model': 'KNN', 'n_neighbors_knn': 7, 'weights_knn': 'uniform'}. Best is trial 13 with value: 3.0520134758073443.
🏃 View run delicate-loon-41 at: https://dags

In [45]:
# view average error for each meta-model type
print("--- AVERAGE ERROR BY META-LEARNER ---")
print(
    study_stacking.trials_dataframe()
    .groupby(by="params_model")["value"]
    .mean()
    .sort_values()
)


--- AVERAGE ERROR BY META-LEARNER ---
params_model
LR     2.988322
KNN    3.172951
DT     3.321332
Name: value, dtype: float64


In [46]:
# build the final production stacking regressor
stacking_reg = StackingRegressor(
    estimators=[("rf", best_rf), ("xgb", best_xgb)],
    final_estimator=LinearRegression(),
    cv=5,
    n_jobs=-1,
)

# wrap in TransformedTargetRegressor to predict in original minutes
final_model = TransformedTargetRegressor(regressor=stacking_reg, transformer=pt)

# train the final model on training data
print("Fitting the final production model...")
final_model.fit(X_train_trans, y_train)
print("Done!")


Fitting the final production model...
Done!


In [47]:
# predictions
y_train_pred = final_model.predict(X_train_trans)
y_test_pred = final_model.predict(X_test_trans)

# MAE (in minutes)
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

# R2 scores
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# 3-Fold Cross-Validation on the final ensemble
cv_scores = cross_val_score(
    final_model,
    X_train_trans,
    y_train,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
cv_mean_mae = -cv_scores.mean()

print("=" * 45)
print("🏆 FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 45)
print(f"Train MAE:               {train_mae:.2f} minutes")
print(f"Test MAE:                {test_mae:.2f} minutes")
print(f"Train R2:                {train_r2:.2f}")
print(f"Test R2:                 {test_r2:.2f}")
print(f"3-Fold Cross-Val MAE:    {cv_mean_mae:.2f} minutes")
print("=" * 45)


c:\Users\ARYAN PRAJAPATI\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning:

X has feature names, but LinearRegression was fitted without feature names

c:\Users\ARYAN PRAJAPATI\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning:

X has feature names, but LinearRegression was fitted without feature names



🏆 FINAL MODEL PERFORMANCE SUMMARY
Train MAE:               2.85 minutes
Test MAE:                2.99 minutes
Train R2:                0.86
Test R2:                 0.84
3-Fold Cross-Val MAE:    3.04 minutes


In [49]:
import os
import joblib

# 1. Ensure models directory exists
os.makedirs("models", exist_ok=True)

# 2. Save both artifacts to disk (used directly by app.py)
joblib.dump(preprocessor, "models/preprocessor.joblib")
joblib.dump(final_model, "models/final_model.joblib")
print("Saved models locally to 'models/' folder!")

# 3. Log metrics, parameters, and upload model artifacts to DagsHub MLflow
with mlflow.start_run(run_name="Final_Production_Estimator"):
  # tags
  mlflow.set_tag("model_type", "StackingRegressor (RF + XGB + LR)")

  # log parameters
  mlflow.log_params(stacking_reg.get_params())

  # log metrics
  mlflow.log_metric("train_mae", train_mae)
  mlflow.log_metric("test_mae", test_mae)
  mlflow.log_metric("train_r2", train_r2)
  mlflow.log_metric("test_r2", test_r2)
  mlflow.log_metric("cv_score", cv_mean_mae)

  # upload the model artifact to DagsHub safely
  mlflow.log_artifact("models/final_model.joblib", artifact_path="model")
  mlflow.log_artifact("models/preprocessor.joblib", artifact_path="preprocessor")

print("Successfully logged final run and uploaded artifacts to DagsHub!")


Saved models locally to 'models/' folder!
🏃 View run Final_Production_Estimator at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/17568d58e5ba4df58146222be50b8a83
🧪 View experiment at: https://dagshub.com/aryann13/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4
Successfully logged final run and uploaded artifacts to DagsHub!
